# 面试问题：模型上线怎样做 shadow、canary、灰度放量和自动回滚？

**一句话回答**：先校验模型/特征/输出 schema 与离线门禁；shadow 复制真实请求但不影响用户，检查兼容和成本；canary 用稳定 hash 做 sticky 小流量，对质量、错误率、延迟和资源 guardrail 做分阶段统计决策。发布状态机只允许单向晋级或立即回滚，配置和观测窗口版本化，回滚切换路由指针而不是临时重建模型。

下面用 NumPy 实现稳定分流、guardrail 差值、序贯多次查看校正和发布状态机。

In [ ]:
from dataclasses import dataclass
from statistics import NormalDist
import hashlib, json, math
import numpy as np

SEED88=8801; rng88=np.random.default_rng(SEED88); NORMAL88=NormalDist()
assert SEED88==8801
assert NORMAL88.cdf(0)==.5
assert hashlib.sha256(b"control").hexdigest()!=hashlib.sha256(b"candidate").hexdigest()

## 1. 发布前先验证 artifact 合同

artifact 包含模型权重摘要、代码、特征 schema、pre/post-processing、训练数据水位、阈值与资源需求。candidate 输出字段可新增可选项，不能删除或改变现有字段语义；特征缺失策略必须明确。

离线指标通过只是进入 shadow 的条件，不代表可以直接全量。

In [ ]:
@dataclass(frozen=True)
class Artifact88:
    version:str; input_schema:tuple; output_schema:tuple; weights_sha:str; max_memory_gb:float
    def __post_init__(self):
        if not self.version or len(self.weights_sha)!=64 or self.max_memory_gb<=0: raise ValueError("artifact_contract")
control88=Artifact88("m1",("x1","x2"),("score","label"),hashlib.sha256(b"w1").hexdigest(),4)
candidate88=Artifact88("m2",("x1","x2"),("score","label","explanation"),hashlib.sha256(b"w2").hexdigest(),4.5)
def compatible88(old,new): return old.input_schema==new.input_schema and set(old.output_schema).issubset(new.output_schema)
assert compatible88(control88,candidate88)
assert control88.weights_sha!=candidate88.weights_sha
try: Artifact88("bad",(),(),"x",0); raise AssertionError("invalid artifact accepted")
except ValueError as e: assert str(e)=="artifact_contract"

## 2. 稳定 hash 分流与实验单位

分流单位应与干扰边界一致，常用 user/account/tenant，而不是每请求随机；同一主体在同一 rollout salt 下保持 sticky。salt 绑定实验版本，避免与其他实验完全相关。internal、机器人和不支持 candidate 的请求应在 hash 前过滤。

百分比从 hash 的均匀区间判断，不能用语言运行时随机 hash，因为进程重启可能变化。

In [ ]:
def bucket88(subject,salt,buckets=10000):
    if not subject or not salt or buckets<1: raise ValueError("assignment_contract")
    return int.from_bytes(hashlib.blake2b(f"{salt}|{subject}".encode(),digest_size=8).digest(),"big")%buckets
def assigned88(subject,salt,percent): return bucket88(subject,salt)<round(percent*100)
subjects88=[f"user-{i}" for i in range(20000)]; assigned1_88=[assigned88(x,"rollout-m2",1) for x in subjects88]
assert .007<np.mean(assigned1_88)<.013
assert all(assigned88(x,"rollout-m2",1)==assigned88(x,"rollout-m2",1) for x in subjects88[:100])
assert any(assigned88(x,"rollout-m2",1)!=assigned88(x,"other-salt",1) for x in subjects88)

## 3. Shadow 验证兼容、成本和差异，不验证用户因果效果

shadow 同时运行 control/candidate，但只返回 control。可比较输出分布、崩溃率、延迟、显存和特定样本差异；不能据 shadow 点击判断 candidate 业务提升，因为用户没看到 candidate。还要避免 shadow 产生写副作用、重复调用外部工具或污染 cache。

这里构造配对请求延迟，配对差值比独立比较方差更小。

In [ ]:
n_shadow88=2000; base_latency88=rng88.lognormal(3.4,.25,n_shadow88); control_latency88=base_latency88+rng88.normal(0,1,n_shadow88); candidate_latency88=base_latency88+2+rng88.normal(0,1,n_shadow88)
paired_delta88=candidate_latency88-control_latency88
shadow_report88={"mean_delta":float(paired_delta88.mean()),"p95_control":float(np.quantile(control_latency88,.95)),"p95_candidate":float(np.quantile(candidate_latency88,.95)),"crash_candidate":0}
assert 1.8<shadow_report88["mean_delta"]<2.2
assert shadow_report88["p95_candidate"]>shadow_report88["p95_control"]
assert shadow_report88["crash_candidate"]==0

## 4. Canary guardrail：效果、错误和非劣效延迟

对 canary/control 分别计算均值差和标准误。业务质量可要求 candidate 改善；错误率、延迟和成本通常做 non-inferiority：candidate 不得比 control 差超过 margin。统计单位必须是分流主体，不能把同一用户的请求当独立样本。

这里用正态近似构造 95% 区间，生产可用 cluster bootstrap 或实验平台的稳健方差。

In [ ]:
def diff_ci88(control,candidate,alpha=.05):
    a=np.asarray(control,float); b=np.asarray(candidate,float)
    if len(a)<2 or len(b)<2: raise ValueError("metric_contract")
    effect=float(b.mean()-a.mean()); se=math.sqrt(a.var(ddof=1)/len(a)+b.var(ddof=1)/len(b)); z=NORMAL88.inv_cdf(1-alpha/2); return effect,(effect-z*se,effect+z*se),se
control_quality88=rng88.normal(.70,.12,1200); candidate_quality88=rng88.normal(.725,.12,1200)
quality_effect88,quality_ci88,quality_se88=diff_ci88(control_quality88,candidate_quality88)
control_error88=rng88.binomial(1,.01,5000); candidate_error88=rng88.binomial(1,.011,5000)
error_effect88,error_ci88,_=diff_ci88(control_error88,candidate_error88)
assert quality_effect88>0 and quality_ci88[0]>0
assert error_ci88[1]<.01
assert quality_se88>0 and error_effect88<.01

## 5. 多次查看需要 alpha spending

每小时看一次普通 0.05 p-value 并“显著就放量”会增加假阳性。教学版用 Bonferroni：计划最多 `L` 次 look，每次使用 `alpha/L`；保守但清晰。更高效可用 group sequential boundary 或 anytime-valid e-value。

look 次数、窗口和停止规则必须发布前写入 manifest，不能看完结果再改。

In [ ]:
def sequential_decision88(effect,se,look,max_looks,overall_alpha=.05):
    if not 1<=look<=max_looks or se<=0: raise ValueError("sequential_contract")
    local_alpha=overall_alpha/max_looks; z=NORMAL88.inv_cdf(1-local_alpha/2); lo=effect-z*se; hi=effect+z*se
    return {"promote":lo>0,"harm":hi<0,"interval":(lo,hi),"local_alpha":local_alpha}
seq88=sequential_decision88(quality_effect88,quality_se88,3,5)
assert math.isclose(seq88["local_alpha"],.01)
assert seq88["interval"][0]<quality_effect88<seq88["interval"][1]
assert not (seq88["promote"] and seq88["harm"])

## 6. 发布状态机与自动回滚

合法路径：`DRAFT -> SHADOW -> CANARY_1 -> CANARY_5 -> CANARY_25 -> FULL`；任何在线阶段都可进入 `ROLLED_BACK`。每次晋级必须满足最小样本、观察时长、质量与全部 guardrail；自动回滚优先于等待人工。旧 artifact 保持 warm，回滚只是原子路由切换。

状态迁移带 expected generation，防止两个控制器并发晋级。

In [ ]:
class Rollout88:
    order=("DRAFT","SHADOW","CANARY_1","CANARY_5","CANARY_25","FULL")
    def __init__(self): self.state="DRAFT"; self.generation=0; self.history=[]
    def transition(self,target,expected_generation,guards_ok):
        if expected_generation!=self.generation: return False,"stale_controller"
        if target=="ROLLED_BACK" and self.state!="DRAFT": allowed=True
        else: allowed=self.state in self.order and self.order.index(self.state)+1<len(self.order) and self.order[self.order.index(self.state)+1]==target and guards_ok
        if not allowed: return False,"invalid_transition"
        self.history.append((self.state,target,self.generation)); self.state=target; self.generation+=1; return True,"applied"
rollout88=Rollout88(); assert rollout88.transition("SHADOW",0,True)[0]
assert not rollout88.transition("CANARY_5",1,True)[0] and rollout88.transition("CANARY_1",1,True)[0]
assert rollout88.transition("ROLLED_BACK",2,False)[0] and rollout88.state=="ROLLED_BACK"

## 7. 组合 guardrail 与故障注入

决策不是“主指标显著就上线”。示例要求质量区间下界为正、错误率差上界低于 1pp、shadow p95 增量低于 5ms、无 schema/crash。任一硬 guardrail 失败立即回滚；软指标只阻止晋级。

故障注入应覆盖模型加载失败、特征缺失、超时、OOM、依赖降级和观测数据延迟。

In [ ]:
def guards88(quality_ci,error_ci,shadow_report):
    checks={"quality":quality_ci[0]>0,"error_noninferior":error_ci[1]<.01,"latency":shadow_report["p95_candidate"]-shadow_report["p95_control"]<5,"no_crash":shadow_report["crash_candidate"]==0}
    return all(checks.values()),checks
ok88,checks88=guards88(quality_ci88,error_ci88,shadow_report88)
assert ok88 and all(checks88.values())
bad_shadow88=dict(shadow_report88,p95_candidate=shadow_report88["p95_control"]+20)
assert not guards88(quality_ci88,error_ci88,bad_shadow88)[0]
assert guards88(quality_ci88,error_ci88,bad_shadow88)[1]["latency"] is False

## 8. 观测、manifest 与面试收束

dashboard 按 model version、rollout bucket、tenant/region、输入长度和缓存命中切片；路由日志记录 assignment salt/bucket，但不记录敏感输入。观测延迟时暂停晋级，不能把“没数据”当通过。manifest 保存阶段比例、最小样本/时长、look 计划、margin 和回滚 artifact。

回答闭环：artifact/schema → sticky hash → shadow → canary 因果指标/guardrail → sequential rule → 状态机 → warm rollback → 全链路审计。

In [ ]:
manifest88={"schema":1,"control":control88.version,"candidate":candidate88.version,"salt":"rollout-m2","stages":[0,1,5,25,100],"max_looks":5,"error_margin":.01,"latency_margin_ms":5,"rollback":control88.version}
raw88=json.dumps(manifest88,sort_keys=True,separators=(",",":")); digest88=hashlib.sha256(raw88.encode()).hexdigest()
assert len(digest88)==64 and manifest88["stages"]==sorted(manifest88["stages"])
assert manifest88["rollback"]==control88.version and manifest88["candidate"]!=manifest88["control"]
forged88=dict(manifest88,error_margin=.1)
assert hashlib.sha256(json.dumps(forged88,sort_keys=True,separators=(",",":")).encode()).hexdigest()!=digest88
print({"quality_effect":round(quality_effect88,4),"error_effect":round(error_effect88,4),"guard_ok":ok88,"sha":digest88[:12]})

## 9. 参考与练习

练习：加入分层 tenant 随机化；实现 cluster bootstrap；模拟指标延迟和控制器并发；设计 FULL 后 24 小时 soak 与自动回退；区分模型回滚和特征回滚。

参考：[Google SRE Workbook：Canarying Releases](https://sre.google/workbook/canarying-releases/)、[Rules of ML：模型上线与监控](https://developers.google.com/machine-learning/guides/rules-of-ml)、[Always Valid Inference](https://arxiv.org/abs/1512.04922)。